In [3]:
import pandas as pd
import sys
import os

# 1. Step UP one folder from 'notebooks' to the main project root, then into 'src'
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir) 
src_path = os.path.join(parent_dir, 'src')
sys.path.append(src_path)

# Import the functions
from hypothesis_tests import test_numerical_kpi, test_categorical_kpi, interpret_p_value

# --- UPDATE THESE COLUMN NAMES TO MATCH YOUR EXACT DATASET ---
COLUMN_NAMES = {
    'premium': 'TotalPremium',
    'claims': 'TotalClaims',
    'province': 'Province',
    'zipcode': 'PostalCode',
    'gender': 'Gender'
}
# ----------------------------------------------------------------

print("Loading data...")
# 2. Tell pandas to step UP one folder to find the 'data' directory
data_path = os.path.join(parent_dir, 'data', 'MachineLearningRating_v3_cleaned.txt')
df = pd.read_csv(data_path, sep='|', low_memory=False)

print("Engineering KPIs...")
# Margin = Premium - Claims
df['Margin'] = df[COLUMN_NAMES['premium']] - df[COLUMN_NAMES['claims']]

# Claim Frequency = 1 if claim occurred, 0 otherwise
df['Claim_Frequency'] = (df[COLUMN_NAMES['claims']] > 0).astype(int)

# Claim Severity = Claim amount (only for rows where a claim actually occurred)
df_with_claims = df[df['Claim_Frequency'] == 1]


print("\n--- HYPOTHESIS TESTING RESULTS ---")

# ---------------------------------------------------------
# Test 1: Risk differences across provinces (Claim Frequency)
# ---------------------------------------------------------
top_provinces = df[COLUMN_NAMES['province']].value_counts().index
prov_A, prov_B = top_provinces[0], top_provinces[1]

group_a = df[df[COLUMN_NAMES['province']] == prov_A]['Claim_Frequency']
group_b = df[df[COLUMN_NAMES['province']] == prov_B]['Claim_Frequency']

stat_1, p_val_1 = test_categorical_kpi(group_a, group_b)
print(f"\n1. Provinces ({prov_A} vs {prov_B}) - KPI: Claim Frequency [Chi-Squared]")
print(interpret_p_value(p_val_1))


# ---------------------------------------------------------
# Test 2: Risk differences between zip codes (Claim Frequency)
# ---------------------------------------------------------
top_zips = df[COLUMN_NAMES['zipcode']].value_counts().index
zip_A, zip_B = top_zips[0], top_zips[1]

group_a = df[df[COLUMN_NAMES['zipcode']] == zip_A]['Claim_Frequency']
group_b = df[df[COLUMN_NAMES['zipcode']] == zip_B]['Claim_Frequency']

stat_2, p_val_2 = test_categorical_kpi(group_a, group_b)
print(f"\n2. Zip Codes ({zip_A} vs {zip_B}) - KPI: Claim Frequency [Chi-Squared]")
print(interpret_p_value(p_val_2))


# ---------------------------------------------------------
# Test 3: Margin differences between zip codes
# ---------------------------------------------------------
group_a = df[df[COLUMN_NAMES['zipcode']] == zip_A]['Margin']
group_b = df[df[COLUMN_NAMES['zipcode']] == zip_B]['Margin']

stat_3, p_val_3 = test_numerical_kpi(group_a, group_b)
print(f"\n3. Zip Codes ({zip_A} vs {zip_B}) - KPI: Margin [T-Test]")
print(interpret_p_value(p_val_3))


# ---------------------------------------------------------
# Test 4: Risk differences between Women and Men (Claim Frequency)
# ---------------------------------------------------------
genders = df[COLUMN_NAMES['gender']].unique()
if len(genders) >= 2:
    gen_A, gen_B = genders[0], genders[1]
    group_a = df[df[COLUMN_NAMES['gender']] == gen_A]['Claim_Frequency']
    group_b = df[df[COLUMN_NAMES['gender']] == gen_B]['Claim_Frequency']

    stat_4, p_val_4 = test_categorical_kpi(group_a, group_b)
    print(f"\n4. Gender ({gen_A} vs {gen_B}) - KPI: Claim Frequency [Chi-Squared]")
    print(interpret_p_value(p_val_4))
else:
    print("\n4. Gender - Could not test. Less than 2 genders found in dataset.")

Loading data...
Engineering KPIs...

--- HYPOTHESIS TESTING RESULTS ---

1. Provinces (Gauteng vs Western Cape) - KPI: Claim Frequency [Chi-Squared]
p-value = 0.00000 (< 0.05): Reject H0. The difference IS statistically significant.

2. Zip Codes (2000 vs 122) - KPI: Claim Frequency [Chi-Squared]
p-value = 0.05788 (>= 0.05): Fail to reject H0. The difference is NOT statistically significant.

3. Zip Codes (2000 vs 122) - KPI: Margin [T-Test]
p-value = 0.24446 (>= 0.05): Fail to reject H0. The difference is NOT statistically significant.

4. Gender (Not specified vs Male) - KPI: Claim Frequency [Chi-Squared]
p-value = 0.01669 (< 0.05): Reject H0. The difference IS statistically significant.


## Hypothesis Testing Results & Business Recommendations

### Summary Table

| Hypothesis (H₀) | KPI Tested | Test Used | p-value | Decision |
| :--- | :--- | :--- | :--- | :--- |
| No risk differences across provinces | Claim Frequency | Chi-Squared | 0.00000 | **Reject H₀** |
| No risk differences between zip codes | Claim Frequency | Chi-Squared | 0.05788 | **Fail to Reject** |
| No margin differences between zip codes | Margin | T-Test | 0.24446 | **Fail to Reject** |
| No risk differences between Gender | Claim Frequency | Chi-Squared | 0.01669 | **Reject H₀** |

---

### Business Recommendations
* **Provinces (Rejected):** The data shows definitive regional risk profiles (Gauteng vs Western Cape). We recommend implementing a regional risk adjustment multiplier to our premiums, increasing baseline costs in higher-frequency provinces.
* **Gender Data (Rejected):** We found a statistically significant risk difference when analyzing gender categories (specifically comparing 'Male' against 'Not specified' profiles). We recommend adjusting pricing based on demographic risk pooling, while also investigating our data collection methods to reduce the number of 'Not specified' entries.